In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## Import du dataset

In [ ]:
df = pd.read_csv(
    "../data/raw_data.csv",
    sep=";",
    encoding="utf-8-sig",
    index_col=False
)

df = df.sort_values(by=["Date", "Heure"], ascending=True).reset_index(drop=True)

## Aperçu général

In [ ]:
df.head(10)

In [ ]:
df.columns

In [ ]:
df.describe()

### Colonnes constantes

Colonnes qui ne contiennent qu'une seule valeur sur toute la série.

In [ ]:
# Les colonnes qui ne contiennent qu'une seule même valeur pour toutes ses observations
df.columns[df.nunique() == 1]

## Nettoyage et préparation

On consolide ici le typage, les filtres et la création des colonnes temporelles dérivées utilisées par les analyses.

In [ ]:
# Les données définitives fonctionnent par pas de 30 min :
# on retire les relevés 00:15 / 00:45 (NaN sur la consommation)
df = df.dropna(subset=["Consommation (MW)"])

# Conversion des colonnes d'échanges commerciaux (contiennent parfois des chaînes non numériques)
cols_ech = [
    "Ech. comm. Angleterre (MW)",
    "Ech. comm. Italie (MW)",
    "Ech. comm. Espagne (MW)",
    "Ech. comm. Suisse (MW)",
    "Ech. comm. Allemagne-Belgique (MW)",
]
df[cols_ech] = df[cols_ech].apply(pd.to_numeric, errors="coerce")

# Colonnes temporelles dérivées (réutilisées par toutes les analyses)
df["Date"] = pd.to_datetime(df["Date"])
df["Timestamp"] = pd.to_datetime(df["Date"].astype(str) + " " + df["Heure"].astype(str))
df["Heure"] = pd.Categorical(df["Heure"], categories=sorted(df["Heure"].unique()), ordered=True)
df["Année"] = df["Date"].dt.to_period("Y")
df["AnnéeMois"] = df["Date"].dt.to_period("M")
df["Mois"] = df["Date"].dt.month
df["Jour"] = df["Date"].dt.dayofweek

# Production totale (somme des filières principales)
energy_cols_all = [
    "Fioul (MW)", "Charbon (MW)", "Gaz (MW)", "Nucléaire (MW)",
    "Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)", "Bioénergies (MW)",
]
df["Production (MW)"] = df[energy_cols_all].sum(axis=1)

# Période couverte (utilisée dans les titres d'analyses)
year_min = int(df["Date"].dt.year.min())
year_max = int(df["Date"].dt.year.max())
period_label = f"{year_min}-{year_max}"


In [ ]:
df.head(10)

## Qualité & structure des données

### Valeurs non numériques (`"ND"`, etc.)

Certaines colonnes de production (ex. `Gaz - Cogénération`) contiennent des chaînes `"ND"` qui empêchent leur conversion automatique en numérique.

In [ ]:
object_cols = df.select_dtypes(include="object").columns.tolist()

nd_rows = []
for col in object_cols:
    s = df[col]
    converted = pd.to_numeric(s, errors="coerce")
    non_numeric_mask = s.notna() & converted.isna()
    n = int(non_numeric_mask.sum())
    if n == 0:
        continue
    examples = s[non_numeric_mask].value_counts().head(5)
    nd_rows.append({
        "colonne": col,
        "nb_non_numériques": n,
        "ratio %": round(n / len(df) * 100, 2),
        "valeurs": ", ".join(f"{v!r} ({c})" for v, c in examples.items()),
    })

nd_df = pd.DataFrame(nd_rows).sort_values("nb_non_numériques", ascending=False)
nd_df

In [ ]:
years_nd = df[df["Gaz - Cogénération (MW)"] == "ND"]["Année"]
years_not_nd = df[df["Gaz - Cogénération (MW)"] != "ND"]["Année"]

print(f'Les données pour le gaz - cogénération sont indisponibles pour les années {", ".join(years_nd.unique().astype(str))} et disponibles pour les années {", ".join(years_not_nd.unique().astype(str))}')

### Continuité temporelle

Les données définitives sont au pas de 30 minutes. On vérifie qu'il n'y a pas de trou dans la série après le `dropna`.

In [ ]:
expected = pd.date_range(df["Timestamp"].min(), df["Timestamp"].max(), freq="30min")
actual = pd.DatetimeIndex(df["Timestamp"])

missing_ts = expected.difference(actual)
duplicates = actual[actual.duplicated()]

print(f"Période couverte     : {expected.min()} → {expected.max()}")
print(f"Pas attendus (30min) : {len(expected):,}")
print(f"Pas présents         : {len(actual):,}")
print(f"Pas manquants        : {len(missing_ts):,} ({len(missing_ts)/len(expected)*100:.3f} %)")
print(f"Pas dupliqués        : {len(duplicates):,}")

if len(missing_ts) > 0:
    missing_by_year = pd.Series(missing_ts.year).value_counts().sort_index()
    print("\nRépartition des pas manquants par année :")
    print(missing_by_year.to_string())

    gaps = pd.Series(missing_ts).diff().dt.total_seconds().div(60)
    consecutive = (gaps == 30).sum()
    print(f"\nPas manquants consécutifs (suivant immédiatement un autre manquant) : {consecutive}")


### Valeurs manquantes par colonne et par année

On visualise quand chaque colonne commence à être renseignée pour identifier les ruptures d'historique (nouvelles technologies tracées tardivement, changements de périmètre).

In [ ]:
import seaborn as sns

missing_flags = df.isna().astype(int)
missing_flags["Année"] = df["Année"].astype(str)
miss_by_year = missing_flags.groupby("Année").mean()
miss_by_year = miss_by_year.loc[:, miss_by_year.max() > 0].sort_index()

fig, ax = plt.subplots(figsize=(14, max(4, 0.35 * miss_by_year.shape[1])))
sns.heatmap(
    miss_by_year.T,
    cmap="Reds",
    vmin=0, vmax=1,
    cbar_kws={"label": "Ratio de manquants"},
    annot=True, fmt=".2f", annot_kws={"size": 8},
    ax=ax,
)
ax.set_title("Ratio de valeurs manquantes par colonne et par année")
ax.set_xlabel("Année")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Valeurs négatives sur les filières de production

Les productions (Fioul, Charbon, Nucléaire, Eolien, Solaire…) ne devraient **pas** être négatives. On exclut `Pompage`, `Ech. physiques` et les `Ech. comm. *` qui sont par nature signés (imports/exports), ainsi que `Stockage batterie` / `Déstockage batterie` et `STEP turbinage` qui peuvent l'être en phase de pompage.

In [ ]:
production_cols = [
    "Fioul (MW)", "Charbon (MW)", "Gaz (MW)", "Nucléaire (MW)",
    "Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)", "Bioénergies (MW)",
    "Fioul - TAC (MW)", "Fioul - Autres (MW)",
    "Gaz - TAC (MW)", "Gaz - CCG (MW)", "Gaz - Autres (MW)",
    "Hydraulique - Fil de l'eau + éclusée (MW)", "Hydraulique - Lacs (MW)",
    "Bioénergies - Déchets (MW)", "Bioénergies - Biomasse (MW)", "Bioénergies - Biogaz (MW)",
]
production_cols = [c for c in production_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

neg_rows = []
for col in production_cols:
    valid = df[col].notna().sum()
    n_neg = int((df[col] < 0).sum())
    if n_neg == 0:
        continue
    neg_rows.append({
        "colonne": col,
        "nb_négatifs": n_neg,
        "ratio %": round(n_neg / valid * 100, 3),
        "min": float(df[col].min()),
        "première_occurrence": df.loc[df[col] < 0, "Timestamp"].min(),
    })

neg_df = pd.DataFrame(neg_rows).sort_values("nb_négatifs", ascending=False)
neg_df

## Analyses exploratoires

### Palette et helpers de visualisation

In [ ]:
energy_cols = [
    "Fioul (MW)", "Charbon (MW)", "Gaz (MW)", "Nucléaire (MW)",
    "Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)", "Bioénergies (MW)",
]

colors = [
    "#E63946",
    "#2196F3",
    "#2A9D8F",
    "#F4A261",
    "#8338EC",
    "#FFBE0B",
    "#FB5607",
    "#3A86FF",
    "#06D6A0",
    "#FF006E",
]

def add_column(groupby, label, color):
    ax.fill_between(range(len(groupby)), 0, groupby[label], alpha=0.1, color=color)
    ax.plot(groupby[label], color=color, label=label)


### Production par source

In [ ]:
annually_cols = ["Fioul (MW)", "Charbon (MW)", "Gaz (MW)", "Nucléaire (MW)", "Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)"]
annual_palette = ["#8B0000", "#333333", "#FF8C00", "#FFD700", "#4CAF50", "#FFF176", "#1565C0"]

# 0.5 pour remettre en heure car les relevés sont toutes les 30 min
annually = df.groupby("Année")[annually_cols].sum() * 0.5
annually_twh = annually / 1e6  # Conversion en TWh pour lisibilité

fig, ax = plt.subplots(figsize=(14, 7))
annually_twh.plot(kind="bar", stacked=True, ax=ax, color=annual_palette, width=0.8)

ax.set_title("Répartition annuelle de la production d'énergie en France", fontsize=14, fontweight="bold")
ax.set_xlabel("Année", fontsize=12)
ax.set_ylabel("Production (TWh)", fontsize=12)
ax.set_xticklabels([str(m) for m in annually_twh.index], rotation=45, ha="right")
ax.legend(title="Source", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f} TWh"))
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
monthly_cols = ["Fioul (MW)", "Charbon (MW)", "Gaz (MW)", "Nucléaire (MW)", "Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)"]
monthly_palette = ["#8B0000", "#333333", "#FF8C00", "#FFD700", "#4CAF50", "#FFF176", "#1565C0"]

monthly_avg = df.groupby("Mois")[monthly_cols].mean()

month_labels = ["Jan", "Fév", "Mar", "Avr", "Mai", "Jun", "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]

fig, ax = plt.subplots(figsize=(14, 7))
monthly_avg.plot(kind="bar", stacked=True, ax=ax, color=monthly_palette, width=0.8)

ax.set_title(f"Moyenne mensuelle de la production d'énergie en France ({period_label})", fontsize=14, fontweight="bold")
ax.set_xlabel("Mois", fontsize=12)
ax.set_ylabel("Production moyenne (MW)", fontsize=12)
ax.set_xticklabels(month_labels, rotation=45, ha="right")
ax.legend(title="Source", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} MW"))
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


### Production vs Consommation

In [ ]:
hourly_avg = df.groupby("Heure")[["Consommation (MW)", "Production (MW)"]].mean().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))

add_column(hourly_avg, "Production (MW)", colors[0])
add_column(hourly_avg, "Consommation (MW)", colors[1])

ax.set_xticks(range(0, 48, 4))
ax.set_xticklabels(hourly_avg.index[::4], rotation=45)
ax.set_ylabel("MW")
ax.legend(loc="upper right", bbox_to_anchor=(1, 1))
plt.title(f"Évolution moyenne de la production et de la consommation dans une journée ({period_label})")
plt.tight_layout()
plt.show()


In [ ]:
month_labels = ["Jan", "Fév", "Mar", "Avr", "Mai", "Jun", "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]

monthly_avg = df.groupby("Mois")[["Consommation (MW)", "Production (MW)"]].mean().sort_index()
monthly_avg.index = month_labels

fig, ax = plt.subplots(figsize=(12, 5))

add_column(monthly_avg, "Production (MW)", colors[0])
add_column(monthly_avg, "Consommation (MW)", colors[1])

ax.set_ylabel("MW")
ax.legend(loc="upper right",  bbox_to_anchor=(1, 1))
plt.title(f"Évolution moyenne de la production et de la consommation dans un mois ({period_label})")
plt.tight_layout()
plt.show()

In [ ]:
yearly_avg = df.groupby("Année")[["Consommation (MW)", "Production (MW)"]].mean().reset_index()
year_labels = sorted(df["Année"].unique())

fig, ax = plt.subplots(figsize=(12, 5))

add_column(yearly_avg, "Production (MW)", colors[0])
add_column(yearly_avg, "Consommation (MW)", colors[1])

ax.set_xticks(range(len(year_labels)), year_labels, rotation=45)
ax.set_ylabel("MW")
ax.legend(loc="upper right",  bbox_to_anchor=(1, 1))
plt.title(f"Évolution moyenne de la production et de la consommation dans une année ({period_label})")
plt.tight_layout()
plt.show()

### Prévisions vs consommation réelle

In [ ]:
pred_cols = ["Consommation (MW)", "Prévision J-1 (MW)", "Prévision J (MW)"]
pred_monthly = df.groupby("AnnéeMois")[pred_cols].mean()
pred_monthly.index = pred_monthly.index.astype(str)

df["Erreur J-1"] = df["Prévision J-1 (MW)"] - df["Consommation (MW)"]
df["Erreur J"] = df["Prévision J (MW)"] - df["Consommation (MW)"]

mae_j1, mae_j = df["Erreur J-1"].abs().mean(), df["Erreur J"].abs().mean()
bias_j1, bias_j = df["Erreur J-1"].mean(), df["Erreur J"].mean()
mape_j1 = (df["Erreur J-1"].abs() / df["Consommation (MW)"]).mean() * 100
mape_j = (df["Erreur J"].abs() / df["Consommation (MW)"]).mean() * 100

print(f"Prévision J-1 → MAE : {mae_j1:>7,.0f} MW | Biais : {bias_j1:>+7,.0f} MW | MAPE : {mape_j1:.2f} %")
print(f"Prévision J   → MAE : {mae_j:>7,.0f} MW | Biais : {bias_j:>+7,.0f} MW | MAPE : {mape_j:.2f} %")

err_monthly = df.groupby("AnnéeMois")[["Erreur J-1", "Erreur J"]].apply(lambda g: g.abs().mean())
err_monthly.index = err_monthly.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

ax = axes[0]
ax.plot(pred_monthly.index, pred_monthly["Consommation (MW)"], label="Consommation", color=colors[1], linewidth=2)
ax.plot(pred_monthly.index, pred_monthly["Prévision J-1 (MW)"], label="Prévision J-1", color=colors[0], linestyle="--", alpha=0.8)
ax.plot(pred_monthly.index, pred_monthly["Prévision J (MW)"], label="Prévision J", color=colors[3], linestyle=":", alpha=0.9)
ax.set_ylabel("Moyenne mensuelle (MW)")
ax.set_title("Consommation réelle vs prévisions (moyenne mensuelle)")
ax.legend(loc="upper right")
ax.grid(axis="y", linestyle="--", alpha=0.5)

ax = axes[1]
ax.plot(err_monthly.index, err_monthly["Erreur J-1"], label="MAE J-1", color=colors[0])
ax.plot(err_monthly.index, err_monthly["Erreur J"], label="MAE J", color=colors[3])
ax.axhline(mae_j1, color=colors[0], linestyle="--", alpha=0.3, label=f"MAE J-1 globale ({mae_j1:,.0f})")
ax.axhline(mae_j, color=colors[3], linestyle="--", alpha=0.3, label=f"MAE J globale ({mae_j:,.0f})")
ax.set_ylabel("Erreur absolue moyenne (MW)")
ax.set_title("Évolution de l'erreur de prévision (MAE mensuel)")
ax.legend(loc="upper right", ncol=2)
ax.grid(axis="y", linestyle="--", alpha=0.5)

year_ticks = [(i, label[:4]) for i, label in enumerate(pred_monthly.index) if label.endswith("-01")]
ax.set_xticks([i for i, _ in year_ticks])
ax.set_xticklabels([y for _, y in year_ticks], rotation=45, ha="right")

plt.tight_layout()
plt.show()


### Échanges commerciaux par pays

In [ ]:
df["Ech. comm. (Moyen)"] = df[cols_ech].mean(axis=1)

cols_ech = ["Ech. comm. Angleterre (MW)", "Ech. comm. Italie (MW)", "Ech. comm. Espagne (MW)", "Ech. comm. Suisse (MW)", "Ech. comm. Allemagne-Belgique (MW)", "Ech. comm. (Moyen)"]

exchange_avg = df.groupby("Année")[cols_ech].mean().reset_index()
year_labels = sorted(df["Année"].unique())

fig, axes = plt.subplots(3, 2, figsize=(12, 15), sharey=True)

for ax, (col, color) in zip(axes.flatten(), zip(cols_ech, colors)):
    add_column(exchange_avg, col, color)
    ax.legend(loc="upper right",  bbox_to_anchor=(1, 1))
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(col)
    ax.set_xticks(range(len(year_labels)), year_labels, rotation=45)
    ax.set_ylabel("MW")
    ax.tick_params(labelleft=True)

plt.suptitle(f"Échanges commerciaux par pays ({period_label})")
plt.tight_layout()

### Consommation par jour de la semaine

In [ ]:
jour_labels = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]

daily_avg = df.groupby("Jour")["Consommation (MW)"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(daily_avg["Jour"], daily_avg["Consommation (MW)"], color=colors[:7])
ax.set_xticks(range(7), jour_labels)
ax.set_ylabel("MW")
ax.set_title(f"Consommation moyenne par jour de la semaine ({period_label})")

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
